# Step 5. 리뷰 일별 수집

**목표**: 각 게임의 할인 전/중/후 구간 일별 리뷰 수 수집  
**입력**: `data/discount_history.csv`  
**출력**: `data/review_daily.csv`  
**API**: Steam `appreviews` (무료, 인증 불필요)  
**예상 소요**: 약 20~50분

### 수집 전략
- 게임당 최신 순(newest-first)으로 페이지네이션
- `collect_from` (가장 이른 할인 시작 -30일) 도달 시 즉시 중단 → 불필요한 호출 없음
- 게임당 최대 200페이지(20,000개)로 상한 설정 (초고인기작 무한루프 방지)
- 최종 커버리지 리포트로 데이터 완성도 확인

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
from collections import defaultdict
from tqdm.auto import tqdm

discount_df = pd.read_csv("../data/discount_history.csv")
discount_df["discount_start"] = pd.to_datetime(discount_df["discount_start"])
discount_df["discount_end"]   = pd.to_datetime(discount_df["discount_end"])

print(f"할인 이벤트: {len(discount_df)}개")
print(f"게임 수: {discount_df['appid'].nunique()}개")

## 설정

In [ ]:
MAX_PAGES  = 50    # 게임당 최대 페이지 (× 100개 = 최대 20,000 리뷰)
SLEEP_SEC  = 0.3    # 페이지 간 딜레이 (초)
PRE_DAYS   = 30     # 할인 시작 전 기준 기간
POST_DAYS  = 14     # 할인 종료 후 수집 기간

# 게임별 필요 날짜 범위 계산
game_ranges = (
    discount_df.groupby("appid")
    .agg(
        first_start=("discount_start", "min"),
        last_end=("discount_end", "max"),
        name=("name", "first"),
        genre=("genre_category", "first"),
    )
    .reset_index()
)
game_ranges["collect_from"] = game_ranges["first_start"] - timedelta(days=PRE_DAYS)
game_ranges["collect_to"]   = game_ranges["last_end"]    + timedelta(days=POST_DAYS)
game_ranges["window_days"]  = (game_ranges["collect_to"] - game_ranges["collect_from"]).dt.days

print(f"수집 대상 게임: {len(game_ranges)}개")
print(f"평균 수집 기간: {game_ranges['window_days'].mean():.0f}일")
print(f"최대 200페이지(20,000개) 상한 → 일평균 28개 이하 게임은 완전 커버")

## 헬퍼 함수

In [ ]:
def collect_game_reviews(appid, cutoff_date):
    """
    Steam appreviews API로 cutoff_date 이후 일별 리뷰 수 수집 (최신 → 과거 순).
    cutoff_date 이전 리뷰를 만나면 즉시 중단.

    Returns:
        daily_counts : dict  {date_str: count}
        pages_used   : int
        total        : int   수집된 리뷰 수
        hit_limit    : bool  MAX_PAGES 도달 여부
    """
    daily_counts = defaultdict(int)
    cursor = "*"
    cutoff_ts = int(cutoff_date.timestamp())
    pages_used = 0
    total = 0
    hit_limit = False

    for _ in range(MAX_PAGES):
        params = {
            "json": 1,
            "filter": "recent",
            "language": "all",
            "review_type": "all",
            "purchase_type": "all",
            "num_per_page": 100,
            "cursor": cursor,
            "filter_offtopic_activity": 0,
        }
        try:
            resp = requests.get(
                f"https://store.steampowered.com/appreviews/{appid}",
                params=params,
                timeout=20,
            )
            resp.raise_for_status()
            data = resp.json()
        except Exception:
            break

        reviews = data.get("reviews", [])
        if not reviews:
            break

        new_cursor = data.get("cursor", "")
        if not new_cursor or new_cursor == cursor:
            break
        cursor = new_cursor
        pages_used += 1

        oldest_ts_in_page = float("inf")
        for review in reviews:
            ts = review.get("timestamp_created", 0)
            if ts < oldest_ts_in_page:
                oldest_ts_in_page = ts
            if ts < cutoff_ts:
                continue
            # utcfromtimestamp deprecated → fromtimestamp with tz=utc 사용
            date_str = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d")
            daily_counts[date_str] += 1
            total += 1

        # 이 페이지에서 cutoff 이전 리뷰가 등장했으면 중단
        if oldest_ts_in_page < cutoff_ts:
            break

        time.sleep(SLEEP_SEC)
    else:
        hit_limit = True

    return dict(daily_counts), pages_used, total, hit_limit


print("함수 정의 완료")

## API 연결 테스트

Terraria 1페이지만 확인.

In [ ]:
resp = requests.get(
    "https://store.steampowered.com/appreviews/105600",
    params={"json": 1, "filter": "recent", "num_per_page": 5, "purchase_type": "all"},
    timeout=10
)
data = resp.json()
print(f"status: {resp.status_code}")
print(f"리뷰 수: {len(data.get('reviews', []))}개")
if data.get("reviews"):
    sample = data["reviews"][0]
    ts = sample.get("timestamp_created", 0)
    print(f"최신 리뷰 날짜: {datetime.utcfromtimestamp(ts).strftime('%Y-%m-%d')}")

## 리뷰 수집 실행

게임별 collect_from(가장 이른 할인 시작 -30일) 이후 모든 리뷰 수집.  
초고인기작(Rust, Terraria 등)은 200페이지 상한에 도달할 수 있음 — 커버리지 리포트에서 확인.

In [ ]:
all_rows = []
coverage_report = []

for i, (_, game) in enumerate(game_ranges.iterrows()):
    appid   = int(game["appid"])
    name    = game["name"]
    genre   = game["genre"]
    cutoff  = game["collect_from"].to_pydatetime().replace(tzinfo=timezone.utc)
    needed  = game["collect_from"].strftime("%Y-%m-%d")

    print(f"[{i+1:02d}/{len(game_ranges)}] {name} (필요시작: {needed}) ...", end=" ", flush=True)

    daily, pages, total, hit_limit = collect_game_reviews(appid, cutoff)

    # 커버리지: 수집된 가장 오래된 날짜 vs 필요한 시작 날짜
    if daily:
        earliest = min(daily.keys())
        coverage_ok = earliest <= needed
    else:
        earliest = "없음"
        coverage_ok = False

    flag = "⚠상한도달" if hit_limit else ("✓" if coverage_ok else "△부분")
    print(f"{pages}p / {total:,}개 / {flag}")

    coverage_report.append({
        "appid": appid, "name": name,
        "필요시작": needed,
        "수집시작": earliest,
        "페이지": pages,
        "리뷰수": total,
        "완전커버": coverage_ok,
        "상한도달": hit_limit,
    })

    for date_str, count in daily.items():
        all_rows.append({
            "appid": appid,
            "name": name,
            "genre_category": genre,
            "date": date_str,
            "daily_reviews": count,
        })

print(f"\n수집 완료")
print(f"  총 행 수: {len(all_rows):,}")
print(f"  총 게임: {len(game_ranges)}개")

## 커버리지 리포트

상한도달=True인 게임은 오래된 이벤트에 대해 데이터가 부분적으로 누락될 수 있음.

In [ ]:
cov_df = pd.DataFrame(coverage_report)
print(f"완전 커버: {cov_df['완전커버'].sum()}개 / {len(cov_df)}개")
print(f"상한(200p) 도달: {cov_df['상한도달'].sum()}개")
print()
print(cov_df.to_string(index=False))

## 결과 확인

In [ ]:
result_df = pd.DataFrame(all_rows)
result_df["date"] = pd.to_datetime(result_df["date"])
result_df = result_df.sort_values(["appid", "date"]).reset_index(drop=True)

print(f"Shape: {result_df.shape}")
print(f"\n장르별 수집 일수:")
print(result_df.groupby("genre_category")["date"].count().rename("행 수"))
print(f"\n게임별 일평균 리뷰 수 (상위 10):")
per_game = result_df.groupby("name")["daily_reviews"].mean().sort_values(ascending=False)
print(per_game.head(10).round(1))
result_df.head(10)

## CSV 저장

In [ ]:
output_path = "../data/review_daily.csv"
result_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path}")
print(f"Shape: {result_df.shape}")